# Train Damage Model — Specialized for Small Object Detection

**Differs from generation training:**
- 3 classes only: physical_damage, scratch, screen_defect
- imgsz=1280 (gấp đôi để thấy detail)
- YOLOv11m (medium) thay vì small
- Mosaic OFF, mild augmentation
- box loss weight tăng (small object localization khó)
- 150 epochs, patience=30

**Expected:** mAP@50 = 0.40-0.55 cho damage classes (vs 0.04-0.17 lần trước)

## Setup Kaggle
1. Settings → Accelerator: **GPU T4 x2**
2. Add Data: upload `yolo_dataset_damage.zip` (~208MB)
3. Internet: ON, Persistence: Files only

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install ultralytics

In [ ]:
!pip install -q ultralytics 'numpy<2.1'
import ultralytics
ultralytics.checks()

## 3. Find dataset (Kaggle auto-extracts zip)

In [ ]:
import os, glob

yaml_paths = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
if not yaml_paths:
    raise FileNotFoundError('Không tìm thấy data.yaml')
ORIG_YAML = yaml_paths[0]
DATASET_ROOT = os.path.dirname(ORIG_YAML)
print(f'data.yaml: {ORIG_YAML}')
!cat {ORIG_YAML}

## 4. Fix paths

In [ ]:
import yaml

with open(ORIG_YAML) as f:
    cfg = yaml.safe_load(f)

cfg['path']  = DATASET_ROOT
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
cfg['test']  = 'test/images'

DATA_YAML = '/kaggle/working/data_damage.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"Classes ({cfg['nc']}): {cfg['names']}")
print(f'Path: {cfg["path"]}')

print('\n=== Verify image folders ===')
for split in ['train', 'valid', 'test']:
    p = os.path.join(cfg['path'], split, 'images')
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    status = '✓' if n > 0 else '✗ MISSING'
    print(f'{status} {split}: {n} files')

## 5. Auto-save callback

In [ ]:
import shutil
from ultralytics import YOLO

BACKUP_DIR = '/kaggle/working/model_backup'
os.makedirs(BACKUP_DIR, exist_ok=True)

def on_fit_epoch_end(trainer):
    weights_dir = trainer.save_dir / 'weights'
    for fname in ['best.pt', 'last.pt']:
        src = weights_dir / fname
        if src.exists():
            shutil.copy(src, f'{BACKUP_DIR}/{fname}')
    epoch = trainer.epoch + 1
    metrics = trainer.metrics if trainer.metrics else {}
    print(f'[BACKUP] Epoch {epoch} → {BACKUP_DIR}/best.pt | mAP50={metrics.get("metrics/mAP50(B)", 0):.4f}')

print('Callback registered.')

## 6. Train YOLOv11m Damage Model — KEY CONFIG

**Tối ưu cho small object detection:**
- `imgsz=1280` — gấp đôi resolution để thấy detail
- `model='yolo11m.pt'` — medium variant, capacity lớn hơn small
- `batch=8` — phải giảm vì imgsz lớn (T4 16GB chỉ vừa đủ)
- `mosaic=0.0` — TẮT mosaic (phá small object)
- `box=10.0` — tăng weight bbox loss (localization khó hơn classification)
- `cls=0.3` — giảm cls weight
- `mixup=0.0`, `degrees=10`, `scale=0.3` — augmentation MILD
- `epochs=150`, `patience=30` — train lâu hơn vì khó hội tụ

In [ ]:
model = YOLO('yolo11m.pt')  # ← MEDIUM thay vì small
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

results = model.train(
    data=DATA_YAML,
    epochs=150,                # ← Nhiều hơn vì khó học
    imgsz=1280,                # ← KEY: gấp đôi để thấy detail
    batch=8,                   # ← Giảm vì imgsz lớn
    optimizer='AdamW',
    lr0=0.005,                 # ← LR nhỏ hơn cho fine detail
    cos_lr=True,
    patience=30,               # ← Patience lớn hơn
    cache=True,
    amp=True,
    device=0,
    
    # KEY: Augmentation MILD cho small objects
    mosaic=0.0,                # ← TẮT mosaic hoàn toàn
    mixup=0.0,
    hsv_h=0.005,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=10.0,
    translate=0.05,
    scale=0.3,
    fliplr=0.0,                # ← Không flip (camera bump cố định)
    flipud=0.0,
    
    # KEY: Loss weighting bias toward bbox
    box=10.0,                  # ← Tăng box loss (default 7.5)
    cls=0.3,                   # ← Giảm cls (default 0.5)
    dfl=1.5,
    
    project='/kaggle/working/runs',
    name='yolov11m_damage',
    exist_ok=True,
)

print('\nTraining done!')
print(f'Best weights: {results.save_dir}/weights/best.pt')
print(f'Backup: {BACKUP_DIR}/best.pt')

## 7. Evaluate test set

In [ ]:
best_pt = f'{BACKUP_DIR}/best.pt'
model = YOLO(best_pt)

metrics = model.val(data=DATA_YAML, split='test', imgsz=1280)
print(f'\n=== TEST SET METRICS ===')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'\nPer-class mAP@50:')
for i, name in enumerate(cfg['names']):
    print(f'  {name:<20} {metrics.box.maps[i]:.4f}')

## 8. Final save

In [ ]:
FINAL_OUT = '/kaggle/working/final_damage'
os.makedirs(FINAL_OUT, exist_ok=True)

shutil.copy(f'{BACKUP_DIR}/best.pt', f'{FINAL_OUT}/best_damage.pt')
shutil.copy(f'{BACKUP_DIR}/last.pt', f'{FINAL_OUT}/last_damage.pt')

src_dir = '/kaggle/working/runs/yolov11m_damage'
for fname in ['results.png', 'results.csv', 'confusion_matrix.png',
              'confusion_matrix_normalized.png', 'F1_curve.png',
              'PR_curve.png', 'P_curve.png', 'R_curve.png']:
    src = f'{src_dir}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{FINAL_OUT}/{fname}')

print(f'Saved to: {FINAL_OUT}')
!ls -la {FINAL_OUT}
print('\n👉 Click "Save Version" → "Quick Save" để commit')